# Butly LoCoMo Evaluation (Colab Pro)

This notebook is a **thin frontend**: it mounts Drive, prepares the repo and
a local model server, then drives `python -m evals.locomo.cli`. All
evaluation logic lives in `evals/locomo/` — do not add scoring, replay, or
checkpoint code here.

Prerequisites:

* The LoCoMo dataset JSON on your Drive (official data is CC BY-NC 4.0 and is
  **not** bundled with Butly — download it from
  https://github.com/snap-research/locomo yourself).
* Optionally a Hugging Face token in Colab Secrets (`HF_TOKEN`) for gated
  model downloads.

Artifacts (checkpoints included) are written to Drive, so a disconnected
runtime can continue with the **Resume** cell below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Parameters (edit these) ---
REPO_URL = 'https://github.com/unagisann/butly.git'
BRANCH = 'main'
REPO_DIR = '/content/butly'

DRIVE_ROOT = '/content/drive/MyDrive/butly-evals'
DATASET_PATH = f'{DRIVE_ROOT}/data/locomo10.json'
RUN_ID = 'qwen3_14b_colab'   # one run directory per model + attempt

# llama.cpp server (any OpenAI-compatible server works)
MODEL_HF_REPO = 'Qwen/Qwen3-14B-GGUF'
MODEL_HF_FILE = 'Qwen3-14B-Q4_K_M.gguf'
MODEL_NAME = 'qwen3-14b'
SERVER_PORT = 8080

SAMPLE_LIMIT = 1
SESSION_LIMIT = 3
QUESTION_LIMIT = 10

In [ ]:
# --- Clone / update Butly and install dependencies ---
import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# --- Download the model and start an OpenAI-compatible server ---
import os
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN') or '')
except Exception:
    pass  # token is only needed for gated models

!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(MODEL_HF_REPO, MODEL_HF_FILE)

# llama.cpp is one option; vLLM or Ollama work as long as the server speaks
# the OpenAI API. Nothing in the evaluation depends on the server choice.
!apt-get -qq install -y libcurl4-openssl-dev > /dev/null
![ -d /content/llama.cpp ] || (git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp && cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-server -j > /dev/null)

import subprocess, time, urllib.request
server = subprocess.Popen([
    '/content/llama.cpp/build/bin/llama-server',
    '-m', model_path, '--port', str(SERVER_PORT), '-ngl', '99',
])
for _ in range(120):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{SERVER_PORT}/health', timeout=2)
        print('model server is up')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('model server did not become healthy')

In [ ]:
# --- Register the server as a Butly connection + write the eval profile ---
import json, pathlib
user_config = {
    'LLM_CONNECTIONS': [
        {
            'id': 'colab_local',
            'protocol': 'openai_compat',
            'base_url': f'http://127.0.0.1:{SERVER_PORT}/v1',
            'api_key_env': 'COLAB_LOCAL_API_KEY',
            'label': 'Colab local server',
        }
    ]
}
pathlib.Path('user_config.json').write_text(json.dumps(user_config, indent=2))
import os
os.environ['COLAB_LOCAL_API_KEY'] = 'local'  # llama.cpp accepts any key

profile = pathlib.Path('evals/locomo/profiles/full_local.example.yaml').read_text()
profile = profile.replace('qwen3-14b', MODEL_NAME)
pathlib.Path('evals/locomo/profiles/full_local.yaml').write_text(profile)
print('connection + profile ready')

In [ ]:
# --- Run the evaluation (replay -> sleeptime -> QA -> score -> report) ---
!python -m evals.locomo.cli run \
  --dataset "{DATASET_PATH}" \
  --output-dir "{DRIVE_ROOT}/runs" \
  --run-id "{RUN_ID}" \
  --profile evals/locomo/profiles/full_local.yaml \
  --sample-limit {SAMPLE_LIMIT} \
  --session-limit {SESSION_LIMIT} \
  --question-limit {QUESTION_LIMIT}

In [ ]:
# --- Resume after a runtime disconnect (safe to re-run; skips finished work) ---
# Re-run the setup cells above first (mount, clone, server, connection),
# then execute this cell instead of the run cell.
!python -m evals.locomo.cli resume --run-dir "{DRIVE_ROOT}/runs/{RUN_ID}"

In [ ]:
# --- Show the summary ---
from IPython.display import Markdown, display
import pathlib
display(Markdown(pathlib.Path(f'{DRIVE_ROOT}/runs/{RUN_ID}/summary.md').read_text()))